# TATR-Span Ablation Study (E0~E3)

**실행 전 체크리스트**
- [ ] 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
- [ ] 셀을 **위에서 아래로** 순서대로 실행

| ID | 설명 |
|----|------|
| E0 | Baseline TATR (원본 재현) |
| E1 | + Span Attribute Branch |
| E2 | + Hard Grid-Snapping |
| E3 | + Soft Grid-Snapping |

## 0. GPU / 디스크 확인

In [ ]:
!nvidia-smi | head -12
import torch, shutil
print(f"CUDA : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
total, used, free = shutil.disk_usage("/")
print(f"Disk : {free/1e9:.0f} GB free / {total/1e9:.0f} GB total")

## 1. 환경 설정

In [ ]:
!pip install -q pycocotools huggingface_hub
print("Done")

In [ ]:
import os
REPO_DIR = "/content/t1"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Jax0303/t1.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print("Already cloned — pulled latest")

%cd {REPO_DIR}
!git log --oneline -3

## 2. PubTables-1M 다운로드 (Colab 직접 다운, D드라이브 업로드 불필요)

HuggingFace `bsmock/pubtables-1m`에서 직접 받음.  
Colab 인터넷 속도가 빠르므로 21GB도 수 분 이내.

> **SUBSET_MODE = True** (기본값): 이미지 없이 annotations만 받아서 소규모 테스트  
> **SUBSET_MODE = False**: train 이미지(21GB) 포함 전체 다운로드 → 실제 학습

In [ ]:
# ── 모드 설정 ───────────────────────────────────────────────────
# True  → annotations + val images만 (빠른 smoke test, ~1GB)
# False → 전체 train images 포함 (~30GB 추출, 실제 학습용)
SUBSET_MODE = True

TRAIN_MAX = 500  if SUBSET_MODE else None   # 학습 샘플 상한
VAL_MAX   = 100  if SUBSET_MODE else None
EPOCHS    = 3    if SUBSET_MODE else 20
SEEDS     = [42, 43, 44]

DATA_ROOT  = "/content/pubtables"
OUTPUT_DIR = "/content/outputs/ablation"
os.makedirs(DATA_ROOT,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

SRC_DIR  = os.path.join(REPO_DIR, "tatr_base", "src")
CONFIG   = os.path.join(SRC_DIR, "structure_config.json")
TRAIN_PY = os.path.join(SRC_DIR, "main.py")
EVAL_PY  = os.path.join(SRC_DIR, "eval_by_complexity.py")
AGG_PY   = os.path.join(REPO_DIR, "aggregate_results.py")

print(f"SUBSET_MODE : {SUBSET_MODE}")
print(f"EPOCHS      : {EPOCHS}")
print(f"TRAIN_MAX   : {TRAIN_MAX}")

In [ ]:
import tarfile, os
from huggingface_hub import hf_hub_download

HF_REPO = "bsmock/pubtables-1m"
CACHE   = "/content/hf_cache"

def download_and_extract(filename, extract_to):
    """HuggingFace에서 tar.gz 다운로드 후 지정 폴더에 압축 해제.
    
    tar 내부가 flat(폴더 없이 파일만 있음)이므로 extract_to를 명시적으로 지정해야 함.
    extract_structure_dataset.sh 와 동일한 방식.
    """
    marker = os.path.join(DATA_ROOT, f".done_{filename}")
    if os.path.exists(marker):
        print(f"[skip] {filename} 이미 완료")
        return
    print(f"[download] {filename} ...")
    local = hf_hub_download(
        repo_id=HF_REPO,
        filename=filename,
        repo_type="dataset",
        cache_dir=CACHE,
    )
    os.makedirs(extract_to, exist_ok=True)
    print(f"[extract]  → {extract_to}")
    with tarfile.open(local, "r:gz") as tar:
        tar.extractall(extract_to)
    open(marker, 'w').close()
    print(f"[done]     {filename}")

# tar 내부가 flat이므로 각 파일을 올바른 서브폴더에 지정해서 풀어야 함
# (extract_structure_dataset.sh 와 동일한 로직)
tasks = [
    ("PubTables-1M-Structure_Annotations_Train.tar.gz", f"{DATA_ROOT}/train"),
    ("PubTables-1M-Structure_Annotations_Val.tar.gz",   f"{DATA_ROOT}/val"),
    ("PubTables-1M-Structure_Annotations_Test.tar.gz",  f"{DATA_ROOT}/test"),
    ("PubTables-1M-Structure_Filelists.tar.gz",         DATA_ROOT),          # txt 파일들 → root
    ("PubTables-1M-Structure_Images_Val.tar.gz",        f"{DATA_ROOT}/images"),
    ("PubTables-1M-Structure_Images_Test.tar.gz",       f"{DATA_ROOT}/images"),
]

for filename, dest in tasks:
    download_and_extract(filename, dest)

# train 이미지는 SUBSET_MODE=False 일 때만 (21GB)
if not SUBSET_MODE:
    print("[info] train 이미지 다운로드 중 (~21GB) ...")
    download_and_extract("PubTables-1M-Structure_Images_Train.tar.gz",
                         f"{DATA_ROOT}/images")
else:
    print("[info] SUBSET_MODE=True → train 이미지 스킵")
    print("       학습 시 --train_max_size로 샘플 수 제한하고 val 이미지 재사용")

print("\n완료")

In [ ]:
# 다운로드 결과 확인
import os
for sub in ['train', 'val', 'test', 'images']:
    path = os.path.join(DATA_ROOT, sub)
    if os.path.exists(path):
        n = len(os.listdir(path))
        print(f"OK  {path}  ({n:,} files)")
    else:
        print(f"MISSING  {path}")

# SUBSET_MODE에서 train 이미지가 없으면 val/images 폴더를 train에 심볼릭 링크
# (main.py가 data_root/train 폴더에서 이미지를 찾으므로)
if SUBSET_MODE:
    # PubTables-1M은 images/ 폴더에 전체 이미지 통합 저장
    # subset이면 train 이미지 없어도 --train_max_size=500 + images/ 로 동작
    print("\n[info] SUBSET_MODE: train_max_size로 샘플 수 제한")

## 3. Sanity Check (CPU, 데이터 불필요)

In [ ]:
!python {REPO_DIR}/sanity_check.py

## 4. 학습 실행 (E0~E3)

각 셀 독립 실행 가능. 세션 끊기면 섹션 9 참고.

In [ ]:
import subprocess, sys, os, time

def run_one(exp_id, seed, extra_args=()):
    out_dir  = os.path.join(OUTPUT_DIR, exp_id, f"seed{seed}")
    os.makedirs(out_dir, exist_ok=True)
    log_path = os.path.join(out_dir, "run.log")

    cmd = [
        sys.executable, TRAIN_PY,
        "--data_root_dir",  DATA_ROOT,
        "--config_file",    CONFIG,
        "--backbone",       "resnet18",
        "--data_type",      "structure",
        "--mode",           "train",
        "--epochs",         str(EPOCHS),
        "--model_save_dir", out_dir,
        "--metrics_save_filepath", os.path.join(out_dir, "metrics.json"),
        "--device",         "cuda",
        "--seed",           str(seed),
        "--num_workers",    "2",
    ]
    if TRAIN_MAX: cmd += ["--train_max_size", str(TRAIN_MAX)]
    if VAL_MAX:   cmd += ["--val_max_size",   str(VAL_MAX)]
    cmd += list(extra_args)

    print(f"\n{'='*55}")
    print(f"  {exp_id}  seed={seed}  epochs={EPOCHS}")
    print(f"  out : {out_dir}")
    print(f"{'='*55}")

    t0 = time.time()
    with open(log_path, 'w') as log:
        proc = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True,
            cwd=SRC_DIR,  # sys.path.append("../detr") 가 tatr_base/detr 를 가리키도록
        )
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
        proc.wait()

    elapsed = (time.time() - t0) / 60
    ok = proc.returncode == 0
    print(f"\n{'OK' if ok else 'FAIL'} {exp_id} seed={seed} — {elapsed:.1f}분")
    return ok

print("run_one() 준비 완료")

In [ ]:
# E0: Baseline TATR
for seed in SEEDS:
    run_one("E0_baseline", seed, extra_args=("--no_span_branch", "--no_grid_snapping"))

In [ ]:
# E1: + Span Attribute Branch
for seed in SEEDS:
    run_one("E1_span", seed, extra_args=("--span_loss_coef", "0.5", "--no_grid_snapping"))

In [ ]:
# E2: + Hard Grid-Snapping
for seed in SEEDS:
    run_one("E2_snap_hard", seed, extra_args=(
        "--span_loss_coef", "0.5", "--grid_snapping", "hard", "--n_warm", "5"))

In [ ]:
# E3: + Soft Grid-Snapping
for seed in SEEDS:
    run_one("E3_snap_soft", seed, extra_args=(
        "--span_loss_coef", "0.5", "--grid_snapping", "soft", "--n_warm", "5"))

## 5. 복잡도별 평가

In [ ]:
import subprocess, sys
out_csv = os.path.join(OUTPUT_DIR, "results_by_complexity.csv")
subprocess.run(
    [sys.executable, EVAL_PY,
     "--results_dir", OUTPUT_DIR,
     "--data_root",   DATA_ROOT,
     "--xml_subdir",  "test",
     "--output_csv",  out_csv],
    cwd=SRC_DIR, check=True
)
print(f"→ {out_csv}")

## 6. 결과 집계

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, AGG_PY,
     "--results_dir", OUTPUT_DIR,
     "--output_csv",  os.path.join(OUTPUT_DIR, "summary_results.csv")],
    cwd=REPO_DIR, check=True
)

## 7. 결과 시각화

In [ ]:
import pandas as pd
df = pd.read_csv(os.path.join(OUTPUT_DIR, "results_by_complexity.csv"))
display(
    df[df['split']=='all']
    .groupby('experiment_id')[['GriTS_Top','GriTS_Loc','GriTS_Con']]
    .agg(['mean','std']).round(4)
)

In [ ]:
import matplotlib.pyplot as plt, numpy as np

exps   = ['E0_baseline','E1_span','E2_snap_hard','E3_snap_soft']
labels = ['E0\nBaseline','E1\nSpan','E2\nHard','E3\nSoft']
colors = {'simple':'#4C72B0','complex':'#DD8452'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ['GriTS_Loc','GriTS_Top']):
    for split, off in [('simple',-0.2),('complex',0.2)]:
        sub = df[df['split']==split].copy()
        sub[metric] = sub[metric].astype(float)
        means = [sub[sub['experiment_id']==e][metric].mean() for e in exps]
        stds  = [sub[sub['experiment_id']==e][metric].std()  for e in exps]
        ax.bar(np.arange(len(exps))+off, means, 0.35,
               yerr=stds, label=split, color=colors[split], capsize=4, alpha=0.85)
    ax.set_title(metric); ax.set_xticks(range(len(exps)))
    ax.set_xticklabels(labels); ax.set_ylim(0,1)
    ax.legend(); ax.grid(axis='y', alpha=0.3)
fig.suptitle('Simple vs Complex (mean±std, 3 seeds)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'ablation_main.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
kbins = ['k=1','k=2','k=3~4','k>=5']
e0_m, e3_m = [], []
for kb in kbins:
    s0 = df[(df['experiment_id']=='E0_baseline')&(df['k_bin']==kb)]['GriTS_Loc'].astype(float)
    s3 = df[(df['experiment_id']=='E3_snap_soft')&(df['k_bin']==kb)]['GriTS_Loc'].astype(float)
    e0_m.append(s0.mean() if len(s0) else float('nan'))
    e3_m.append(s3.mean() if len(s3) else float('nan'))

x = np.arange(len(kbins))
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(x-0.2, e0_m, 0.35, label='E0 Baseline', color='#4C72B0', alpha=0.85)
ax.bar(x+0.2, e3_m, 0.35, label='E3 Soft-Snap', color='#55A868', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(kbins)
ax.set_ylabel('GriTS_Loc'); ax.set_ylim(0,1)
ax.set_title('GriTS_Loc by Span Complexity: E0 vs E3')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'ablation_kbin.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. 결과 Drive 백업 (세션 끊기기 전 실행)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, time
ts  = time.strftime('%Y%m%d_%H%M')
dst = f"/content/drive/MyDrive/ablation_results_{ts}"
shutil.copytree(OUTPUT_DIR, dst)
print(f"백업 완료 → {dst}")

## 9. 세션 끊긴 후 이어받기

Colab 세션이 초기화되면:
1. 섹션 0~2 실행 (환경 재구성)
2. Drive에서 결과 복원 (아래 셀)
3. 이어받을 실험만 개별 셀 재실행

In [ ]:
# Drive 백업에서 결과 복원
from google.colab import drive
drive.mount('/content/drive')

import shutil, glob
# 가장 최근 백업 폴더 자동 탐색
backups = sorted(glob.glob("/content/drive/MyDrive/ablation_results_*"))
if backups:
    latest = backups[-1]
    print(f"복원 중: {latest}")
    shutil.copytree(latest, OUTPUT_DIR, dirs_exist_ok=True)
    print("복원 완료")
else:
    print("Drive에 백업 없음")

In [ ]:
# 특정 실험만 체크포인트 이어받기
RESUME_EXP  = "E3_snap_soft"
RESUME_SEED = 42
LOAD_PATH   = os.path.join(OUTPUT_DIR, RESUME_EXP, f"seed{RESUME_SEED}", "model.pth")

if os.path.exists(LOAD_PATH):
    run_one(RESUME_EXP, RESUME_SEED, extra_args=(
        "--span_loss_coef", "0.5",
        "--grid_snapping",  "soft",
        "--n_warm",         "5",
        "--model_load_path", LOAD_PATH,
    ))
else:
    print(f"체크포인트 없음: {LOAD_PATH}")

## 부록. 시간 가이드

| 모드 | 다운로드 | 실험 1회 | 전체 12회 |
|------|---------|---------|----------|
| SUBSET (500샘플, 3epoch) | ~1GB, 수분 | ~10분 | ~2시간 |
| FULL (86K샘플, 20epoch)  | ~30GB, 10~20분 | ~3~4시간 | ~40시간 |

**전략 권장:**
1. `SUBSET_MODE=True`로 smoke test → 모든 셀 정상 동작 확인
2. `SUBSET_MODE=False`로 E0 단독 실행 → 풀 학습 시간 측정
3. 전체 12회는 Colab Pro+ A100 세션 권장